# LangGraph behind an MCP server

Build a small LangGraph workflow, expose it as an MCP tool, and call the graph directly for a quick local check. The example uses a deterministic first-sentence summary, so it needs no model credentials. See the [LangGraph topic](https://waggertron.github.io/tech-learning/topics/ai/langgraph/) for the explanation and MCP client setup.

## Install dependencies

Run this in a Python 3.10 or later environment.

In [ ]:
%pip install langgraph "mcp[cli]"

## Define the graph

The state schema describes values passed between nodes. Each node returns the state fields it updates.

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class SummaryState(TypedDict):
    text: str
    summary: str


def summarize(state: SummaryState) -> dict[str, str]:
    first_sentence = state["text"].strip().split(".", maxsplit=1)[0]
    return {"summary": first_sentence.strip()}


builder = StateGraph(SummaryState)
builder.add_node("summarize", summarize)
builder.add_edge(START, "summarize")
builder.add_edge("summarize", END)
graph = builder.compile()

## Invoke the graph

`invoke` starts one graph run and returns its final state.

In [ ]:
graph.invoke(
    {
        "text": "LangGraph organizes work as stateful graphs. Nodes update shared state.",
        "summary": "",
    }
)

## Expose the graph as an MCP tool

The MCP server owns the interface clients discover. Its tool delegates the work to the compiled graph. This cell registers the tool without starting a server process, so the notebook stays interactive. The complete standalone `server.py` version is in the topic page.

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("langgraph-summary")


@mcp.tool()
def summarize_text(text: str) -> str:
    """Summarize text by returning its first sentence."""
    result = graph.invoke({"text": text, "summary": ""})
    return result["summary"]


print("Registered MCP tool: summarize_text")

## Start the server

From a terminal in the same environment, save the complete server code from the topic page as `server.py`, then run `python server.py` for stdio mode. You can also run `mcp dev server.py` to inspect and call the tool during development. The Python process waits for an MCP client over standard input and output.